# 4.4 Task D — Recommendation System\n**Autore:** Studente 4\n**Input:** `outputs/reviews_clean.parquet`, `outputs/listings_clean.parquet`\n**Output:** `models/recommender_svd.pkl`, `metrics/rec_metrics.json`, `outputs/sample_recommendations.json`

In [ ]:
import pandas as pd\nimport numpy as np\nimport json\nimport joblib\nimport os\n\nfrom sklearn.metrics.pairwise import cosine_similarity\nfrom sklearn.feature_extraction.text import TfidfVectorizer\n\ntry:\n    from surprise import Dataset, Reader, KNNBasic, KNNWithMeans, SVD\n    from surprise.model_selection import train_test_split as surprise_split, GridSearchCV as SurpriseGridSearch\n    from surprise.accuracy import rmse, mae\n    SURPRISE_AVAILABLE = True\nexcept ImportError:\n    SURPRISE_AVAILABLE = False\n    print("surprise non installato. Collaborative filtering e SVD non disponibili.")

## 4.4.1 Caricamento Dati

In [ ]:
reviews = pd.read_parquet("outputs/reviews_clean.parquet")\nlistings = pd.read_parquet("outputs/listings_clean.parquet")\n\nprint("Reviews:", reviews.shape)\nprint("Listings:", listings.shape)

## 4.4.2 Costruzione Rating Derivato

In [ ]:
# Opzione B (preferita): review_scores_rating del listing come rating uniforme\nratings = reviews.merge(\n    listings[["id", "review_scores_rating"]],\n    left_on="listing_id", right_on="id", how="inner"\n).dropna(subset=["review_scores_rating"])\n\nratings["rating"] = ratings["review_scores_rating"]\nratings = ratings[["reviewer_id", "listing_id", "rating"]]\n\n# Opzione A (bridge): se esiste sentiment score, prova confronto opzionale\nif os.path.exists("outputs/comment_sentiments.csv"):\n    print("[BRIDGE] Comment sentiments disponibili — Opzione A testabile per confronto.")\nelse:\n    print("[BRIDGE] Nessun sentiment score — uso Opzione B (review_scores_rating).")\n\nprint("Shape matrice rating:", ratings.shape)\nprint("Sparsita:", len(ratings) / (ratings["reviewer_id"].nunique() * ratings["listing_id"].nunique()))

## 4.4.3 Approccio 1 — Content-Based Filtering

In [ ]:
# Vettore listing: descrizione testuale + feature strutturali\n# TODO: aggiungere property_type, neighbourhood, ecc. one-hot encoded\n\nlistings["description_fill"] = listings["description"].fillna("")\ntfidf = TfidfVectorizer(max_features=5000, stop_words="english")\ndesc_tfidf = tfidf.fit_transform(listings["description_fill"])\n\n# Similarita' coseno tra listing\ncos_sim = cosine_similarity(desc_tfidf)\n\ndef content_based_recommend(liked_listing_ids, top_n=5):\n    """Restituisce top-N listing simili a quelli piaciuti."""\n    liked_indices = listings[listings["id"].isin(liked_listing_ids)].index.tolist()\n    if not liked_indices:\n        return []\n    mean_sim = cos_sim[liked_indices].mean(axis=0)\n    # Escludi quelli gia' visti\n    for idx in liked_indices:\n        mean_sim[idx] = -1\n    top_idx = np.argsort(mean_sim)[::-1][:top_n]\n    return listings.iloc[top_idx][["id", "name", "neighbourhood_cleansed"]].to_dict("records")

## 4.4.4 Approccio 2 — Collaborative Filtering (KNN)

In [ ]:
if SURPRISE_AVAILABLE:\n    reader = Reader(rating_scale=(1, 5))\n    data = Dataset.load_from_df(ratings[["reviewer_id", "listing_id", "rating"]], reader)\n    trainset, testset = surprise_split(data, test_size=0.2, random_state=42)\n\n    knn = KNNWithMeans(k=20, sim_options={"name": "cosine", "user_based": False})\n    knn.fit(trainset)\n    knn_preds = knn.test(testset)\n\n    knn_rmse = rmse(knn_preds, verbose=False)\n    knn_mae = mae(knn_preds, verbose=False)\n    print(f"KNN: RMSE={knn_rmse:.4f}, MAE={knn_mae:.4f}")\nelse:\n    knn_rmse, knn_mae = None, None

## 4.4.5 Approccio 3 — SVD (Matrix Factorization)

In [ ]:
if SURPRISE_AVAILABLE:\n    svd = SVD(random_state=42)\n    svd.fit(trainset)\n    svd_preds = svd.test(testset)\n\n    svd_rmse = rmse(svd_preds, verbose=False)\n    svd_mae = mae(svd_preds, verbose=False)\n    print(f"SVD: RMSE={svd_rmse:.4f}, MAE={svd_mae:.4f}")\nelse:\n    svd_rmse, svd_mae = None, None

## 4.4.6 Confronto

In [ ]:
results = [\n    {"approach": "KNN", "RMSE": knn_rmse, "MAE": knn_mae},\n    {"approach": "SVD", "RMSE": svd_rmse, "MAE": svd_mae},\n]\npd.DataFrame(results)

## 5.4 Ottimizzazione SVD

In [ ]:
if SURPRISE_AVAILABLE:\n    param_grid = {\n        "n_factors": [50, 100],\n        "lr_all": [0.002, 0.005],\n        "reg_all": [0.02, 0.1],\n    }\n    gs = SurpriseGridSearch(SVD, param_grid, measures=["rmse"], cv=3, n_jobs=-1)\n    gs.fit(data)\n\n    print("Best RMSE:", gs.best_score["rmse"])\n    print("Best params:", gs.best_params["rmse"])\n\n    best_svd = gs.best_estimator["rmse"]\n    best_svd.fit(trainset)\n    best_preds = best_svd.test(testset)\n    best_rmse = rmse(best_preds, verbose=False)\n    best_mae = mae(best_preds, verbose=False)\nelse:\n    best_svd = None\n    best_rmse, best_mae = None, None

## 4.4.7 Esempio Qualitativo

In [ ]:
# Scegli un utente campione con almeno 3 recensioni\nuser_counts = ratings["reviewer_id"].value_counts()\nsample_user = user_counts[user_counts >= 3].index[0]\n\nliked = ratings[ratings["reviewer_id"] == sample_user]["listing_id"].tolist()\nprint("Utente:", sample_user)\nprint("Listing gia' recensiti:", liked)

In [ ]:
# Top 5 Content-Based\ncb_recs = content_based_recommend(liked, top_n=5)\n\n# Top 5 SVD (se disponibile)\nsvd_recs = []\nif best_svd:\n    all_listings = listings["id"].unique()\n    unseen = [lid for lid in all_listings if lid not in liked]\n    preds = [(lid, best_svd.predict(sample_user, lid).est) for lid in unseen[:100]]\n    preds.sort(key=lambda x: x[1], reverse=True)\n    top_svd = [p[0] for p in preds[:5]]\n    svd_recs = listings[listings["id"].isin(top_svd)][["id", "name", "neighbourhood_cleansed"]].to_dict("records")\n\nsample_output = {\n    "user": int(sample_user),\n    "liked": [int(x) for x in liked],\n    "content_based": cb_recs,\n    "svd": svd_recs\n}\n\nwith open("outputs/sample_recommendations.json", "w", encoding="utf-8") as f:\n    json.dump(sample_output, f, ensure_ascii=False, indent=2)\n\nprint("Esempio qualitativo esportato.")

## Export

In [ ]:
if best_svd:\n    joblib.dump(best_svd, "models/recommender_svd.pkl")\n\nwith open("metrics/rec_metrics.json", "w") as f:\n    json.dump({\n        "results": results,\n        "best_svd": {"RMSE": best_rmse, "MAE": best_mae}\n    }, f, indent=2)\n\nprint("Task D completato e esportato.")